# Model B — Conjunctivitis Binary Classification

This notebook implements and evaluates the **Model B (MobileNetV3-Small)** classifier differentiating Normal ocular surface versus Conjunctivitis.

### Clinical Constraints Enforced:
1. **Never interpret general redness as proof of infectious conjunctivitis**: Differentiates Normal vs. Conjunctivitis using validated multi-disease eye dataset.
2. **Independent model head**: Separate architecture and cross-entropy loss with class reweighting.
3. **Preserve clinical colors**: No solarization or color inversion.

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np

WORKSPACE_ROOT = os.path.abspath('..')
if WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, WORKSPACE_ROOT)

from src.preprocessing.dataset_conjunctivitis import build_conjunctivitis_dataloaders
from src.models.conjunctivitis_model import load_conjunctivitis_checkpoint
from src.evaluation.evaluate import compute_classification_metrics

print('Torch version:', torch.__version__)

## 1. Load Processed Manifests & DataLoaders

In [ ]:
manifest_path = os.path.join(WORKSPACE_ROOT, 'data', 'manifests', 'conjunctivitis_manifest.csv')
df = pd.read_csv(manifest_path)
print(f'Total records in manifest: {len(df)}')
print('Class distribution:', df['class_name'].value_counts().to_dict())

loaders = build_conjunctivitis_dataloaders(
    manifest_csv=manifest_path,
    workspace_root=WORKSPACE_ROOT,
    batch_size=32,
    use_processed_dir=os.path.join(WORKSPACE_ROOT, 'data', 'processed', 'conjunctivitis')
)

## 2. Load Checkpoint and Evaluate

In [ ]:
checkpoint_path = os.path.join(WORKSPACE_ROOT, 'saved_models', 'conjunctivitis', 'best_model.pth')
if os.path.exists(checkpoint_path):
    model = load_conjunctivitis_checkpoint(checkpoint_path)
    print('Loaded checkpoint:', checkpoint_path)
else:
    print('Checkpoint not found. Run scripts/train_conjunctivitis.py first.')

In [ ]:
model.eval()
all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for batch in loaders['test']:
        images = batch['image']
        labels = batch['label']
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.numpy())
        all_probs.extend(probs.numpy())
        all_labels.extend(labels.numpy())

metrics = compute_classification_metrics(all_labels, all_preds, all_probs)
print('=== Test Evaluation Metrics ===')
print(f'Accuracy:  {metrics["accuracy"]:.4f}')
print(f'Precision: {metrics["precision"]:.4f}')
print(f'Recall:    {metrics["recall_sensitivity"]:.4f}')
print(f'F1-Score:  {metrics["f1_score"]:.4f}')
print(f'ROC-AUC:   {metrics["roc_auc"]:.4f}')
print(f'Confusion Matrix:\n', np.array(metrics['confusion_matrix']))